In [ ]:

# %pip install langgraph langchain-core langchain-openai
# %pip install rich python-dotenv

### Defining state

 - Think of this as the "contract" for your entire pipeline.
 - Every node can read ANY field, and return updates to ANY field.

 - Why TypedDict and not a regular dict?
    - It gives you type hints, autocomplete, and catches typos.
    - LangGraph uses it to understand what your state looks like.

In [1]:
from typing import TypedDict, Optional

class ContentState(TypedDict):
    topic: str                          # Input: what to write about
    research_notes: str                 # Filled by Researcher (later)
    draft: str                          # Filled by Writer
    review_score: Optional[int]         # Filled by Reviewer (0-10)
    feedback: str                       # Filled by Reviewer

### Node 1 - The writer

In [3]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()  

api_key = os.getenv("OPEN_ROUTER_KEY")


llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free", 
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "http://localhost:3000", # Required by OpenRouter for some models
        "X-Title": "LangGraph App",
    }
)

def writer_node(state: ContentState) -> dict:
    """
    The Writer node:
    - Reads: 'topic' from state
    - Writes: 'draft' to state
    
    NOTE: We return ONLY the fields we want to update.
    LangGraph merges this into the existing state automatically.
    """
    print("📝 Writer node activated...")
    
    topic = state["topic"]
    
    # If there's feedback from a reviewer, incorporate it
    feedback = state.get("feedback", "")
    
    if feedback:
        # This path will be used when the reviewer sends it back
        prompt = f"""You are a professional content writer.
        
        Rewrite and improve this draft based on the reviewer's feedback.

        TOPIC: {topic}

        PREVIOUS DRAFT:
        {state['draft']}

        REVIEWER FEEDBACK:
        {feedback}

        Write an improved version. Keep it to 2-3 paragraphs."""
    else:
        # First time writing — no feedback yet
        prompt = f"""You are a professional content writer.
        
        Write a short, engaging article about: {topic}

        Keep it to 2-3 paragraphs. Be informative and engaging."""

    # Call the LLM
    response = llm.invoke(prompt)
    
    # response.content is the string output from the LLM
    print(f"📝 Draft generated ({len(response.content)} characters)")
    
    # Return ONLY the fields we're updating
    return {"draft": response.content}


### Reviewer node

In [5]:
def reviewer_node(state: ContentState) -> dict:
    """
    The Reviewer node:
    - Reads: 'draft' and 'topic' from state  
    - Writes: 'review_score' and 'feedback' to state
    
    This uses the LLM to actually evaluate the draft.
    """
    print("🔍 Reviewer node activated...")
    
    draft = state["draft"]
    topic = state["topic"]
    
    prompt = f"""You are a strict content reviewer. 
    
    Review this draft about "{topic}":

    ---
    {draft}
    ---

    Evaluate on these criteria:
    1. Accuracy and relevance to the topic
    2. Clarity and readability  
    3. Engagement and flow
    4. Completeness

    Respond in EXACTLY this format (no extra text):
    SCORE: [number from 1-10]
    FEEDBACK: [your specific feedback for improvement]"""

    response = llm.invoke(prompt)
    response_text = response.content
    
    # Parse the score and feedback from the response
    try:
        lines = response_text.strip().split("\n")
        score_line = [l for l in lines if l.startswith("SCORE:")][0]
        score = int(score_line.split(":")[1].strip().split("/")[0].split()[0])
        
        feedback_start = response_text.find("FEEDBACK:")
        feedback = response_text[feedback_start + 9:].strip() if feedback_start != -1 else "No specific feedback."
    except (ValueError, IndexError):
        # If parsing fails, default to a middle score
        score = 5
        feedback = response_text
    
    # Clamp score to valid range
    score = max(1, min(10, score))
    
    print(f"🔍 Review complete — Score: {score}/10")
    print(f"🔍 Feedback: {feedback[:100]}...")
    
    return {
        "review_score": score,
        "feedback": feedback,
    }

### Building blocks

In [7]:
from langgraph.graph import StateGraph, START, END

# 1. Create the graph builder, passing your State schema
#    This tells LangGraph: "every node will receive and update this shape"
graph_builder = StateGraph(ContentState)

# 2. Add nodes — each gets a name (string) and a function
#    The name is how you reference this node in edges
graph_builder.add_node("writer", writer_node)
graph_builder.add_node("reviewer", reviewer_node)

# 3. Define edges — the flow of execution
#    START is a special constant meaning "entry point"
#    END is a special constant meaning "exit point"
graph_builder.add_edge(START, "writer")       # Start → Writer
graph_builder.add_edge("writer", "reviewer")  # Writer → Reviewer
graph_builder.add_edge("reviewer", END)       # Reviewer → End

# 4. Compile the graph — this validates everything and creates a runnable
#    After this, no more modifications allowed!
graph = graph_builder.compile()

print("Graph compiled successfully!")

Graph compiled successfully!


### Visualize the graph

In [ ]:
# LangGraph can render your graph as an image — super useful for debugging!

from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # If the image rendering fails, print the text version
    print("Graph structure (text):")
    graph.get_graph().print_ascii()
    print(f"\n(Image rendering requires additional deps: {e})")

### Run

In [9]:
# The input must match your State schema
# You only need to provide the fields that are "inputs"
# Other fields will be filled by nodes as execution progresses

initial_state = {
    "topic": "Why Agentic AI is the future of software development",
    "research_notes": "",
    "draft": "",
    "review_score": None,
    "feedback": "",
}

print("🚀 Starting the Content Engine...\n")

# .invoke() runs the graph synchronously from START to END
# It returns the FINAL state after all nodes have executed
final_state = graph.invoke(initial_state)

print("PIPELINE COMPLETE!")


🚀 Starting the Content Engine...

📝 Writer node activated...
📝 Draft generated (1673 characters)
🔍 Reviewer node activated...
🔍 Review complete — Score: 8/10
🔍 Feedback: The draft is accurate, relevant, and well‑structured, but it could be strengthened by: (1) trimming ...
PIPELINE COMPLETE!


In [11]:
from rich import print as rprint
rprint(final_state)

{
    'topic': 'Why Agentic AI is the future of software development',
    'research_notes': '',
    'draft': 'Agentic AI—systems that can perceive, reason, plan, and act autonomously—are reshaping how software 
is built, not just by writing code, but by taking ownership of entire development lifecycles. Unlike traditional 
code assistants that suggest snippets, agentic agents can ingest high‑level product goals, break them into 
executable tasks, orchestrate testing pipelines, and even iterate on designs based on real‑time feedback loops. 
This end‑to‑end agency turns developers from line‑by‑line coders into orchestrators of intelligent collaborators, 
accelerating delivery while reducing the cognitive load of boilerplate and repetitive refactoring.\n\nThe impact is
already visible in rapid‑prototyping environments where agentic AI spins up microservices, generates 
infrastructure‑as‑code, and runs continuous validation without human intervention. By learning from repository 
patterns, issue trackers, and deployment logs, these agents adapt to a team’s conventions, suggest architectural 
improvements, and surface hidden technical debt before it becomes a blocker. The result is a tighter feedback loop 
between idea and production, enabling teams to ship features faster, with higher reliability, and with far less 
manual overhead.\n\nAs the technology matures, agentic AI will become the default “co‑founder” of software 
projects—handling the mundane, augmenting creativity, and freeing engineers to focus on strategic problem‑solving 
and innovation. Embracing this shift isn’t just about adopting a new tool; it’s about redefining the role of the 
developer in an AI‑augmented world, making the future of software development not only faster but fundamentally 
more intelligent.',
    'review_score': 8,
    'feedback': 'The draft is accurate, relevant, and well‑structured, but it could be strengthened by: (1) 
trimming overly long sentences to improve readability; (2) adding a brief note on current limitations or challenges
of agentic AI (e.g., reliability, security, need for oversight) to provide a balanced view; (3) including a 
concrete example or metric (such as a reported reduction in cycle time) to illustrate the impact more vividly; (4) 
varying sentence openings to enhance flow and engagement. These tweaks would make the piece clearer, more complete,
and even more compelling.'
}